In [1]:
import pandas as pd
import ast
from sentence_transformers import SentenceTransformer
from datetime import datetime, date
import json

/home/bsc/bsc093754/miniforge3/envs/datamap_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('/home/bsc/bsc093754/GIT/social-media-data-map/data/processed/aggregate_paths_2026-05-14.csv')
df = df.sample(n=100)

ROOT = '/home/bsc/bsc093754/GIT/social-media-data-map'
save_dir = f'{ROOT}/data/processed'

In [3]:
def combine_list(df):

    df['final_path'] = None

    for ix, row in df.iterrows():
        if row['platform'] == 'Tiktok':
            final_path =  row['path']
            
        else:
            file_list = row['file_path'].split('/')
            path_list = row['path']
        
            
            if isinstance(path_list, str):
                path_list = ast.literal_eval(path_list)

            if isinstance(path_list, list):
                final_path = file_list + path_list

        final_path = '/'.join(final_path)
        df.at[ix, 'final_path'] = final_path        
    
    return df
    


In [4]:
def unique_paths(df):
    platforms = df['platform'].unique()
    df_unique = None
    for p in platforms:
        df_red =  df[df['platform'] == p]
        df_red['final_path'].unique()
        if df_unique is None:
            df_unique = df_red
            
        else:
            df_unique = pd.concat([df_unique, df_red], axis=0, ignore_index=True)
           
    df_unique = df_unique[['platform', 'final_path']]
    return df_unique

In [5]:
def embed_lists(model, df):
    model = SentenceTransformer(model)
    df['final_path_emb'] = None
    
    for ix, row in df.iterrows():
        list_path = row['final_path']
        
        em = model.encode(list_path)
        em = json.dumps(em.tolist())
        

        df.at[ix, 'final_path_emb'] = em

    df.to_csv(f'{save_dir}/path_embeddings_{date.today()}.csv')
    return df
        



In [6]:
model = "/gpfs/projects/bsc100/models/sentence-transformers/all-MiniLM-L6-v2"
df = combine_list(df)
df = unique_paths(df)
df = embed_lists(model, df)

/home/bsc/bsc093754/miniforge3/envs/datamap_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16892.02it/s]
